# 基于MiniLM特征的零样本影评情感识别

In [1]:
# 运行准备：按示例代码5.1～5.2读取并划分IMDB数据，复用示例代码5.61～5.63准备MiniLM嵌入特征
import os
import csv
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sentence_transformers as sentrans

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].tolist()
y = df["label"].tolist()

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at \"{train_file}\" and \"{test_file}\"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_test = df_test["sentence"].tolist()
    y_test = df_test["label"].tolist()
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    pd.DataFrame({'sentence': sents_train, 'label': y_train}).to_csv(train_file, index=False)
    pd.DataFrame({'sentence': sents_test, 'label': y_test}).to_csv(test_file, index=False)

model = sentrans.SentenceTransformer("./fm/all-MiniLM-L6-v2")
label_text = ['A negative movie review is a piece of writing that expresses dissatisfaction with a film. It typically points out perceived flaws such as a weak plot, poor acting, slow pacing, bad dialogue, or technical shortcomings, and it generally advises potential viewers against watching the movie.', 'A positive movie review is a piece of writing that expresses satisfaction with a film. It praises elements such as the story, acting, direction, cinematography, or emotional impact, and it encourages potential viewers to watch the movie.']
label_emb = model.encode(label_text)
x_test = model.encode(sents_test)

/Users/xinzijie/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1570.76it/s]

In [2]:
scores = sentrans.util.cos_sim(x_test, label_emb) #计算影评特征与情感类别特征的余弦相似度
pred = np.argmax(scores, axis=1) #取相似度较大者作为识别结果
pred = pred.cpu().numpy() #从torch.Tensor转为numpy.ndarray
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.856     0.848     0.852       105
           1      0.833     0.842     0.838        95

    accuracy                          0.845       200
   macro avg      0.845     0.845     0.845       200
weighted avg      0.845     0.845     0.845       200

